In [1]:
import duckdb
from pathlib import Path

ROOT = Path(r"C:/Users/DELL/OneDrive/Máy tính/job-market-radar")
ROOT

db_path = Path(ROOT/"data/warehouse/job_market.duckdb")
str(db_path)

'C:\\Users\\DELL\\OneDrive\\Máy tính\\job-market-radar\\data\\warehouse\\job_market.duckdb'

# Inspect Warehouse

In [2]:
con = duckdb.connect(str(db_path))
con.execute("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,job_market,main,adzuna_raw_index,"[download_date, timestamp_local, url, what, wh...","[DATE, TIMESTAMP, VARCHAR, VARCHAR, VARCHAR, B...",False
1,job_market,main,jobs_entry_level_v1,"[source, job_id, created_at, title, company, l...","[VARCHAR, BIGINT, TIMESTAMP WITH TIME ZONE, VA...",False
2,job_market,main,v_jobs,"[source, job_id, created_at, title, company, l...","[VARCHAR, BIGINT, TIMESTAMP WITH TIME ZONE, VA...",False
3,job_market,main,v_replication_groups,"[channel, company, role_signature, postings, d...","[VARCHAR, VARCHAR, VARCHAR, BIGINT, BIGINT, DO...",False


In [3]:
con.execute("PRAGMA table_info('jobs_entry_level_v1')").df()

,cid,name,type,notnull,dflt_value,pk
0,0,source,VARCHAR,False,None,False
1,1,job_id,BIGINT,False,None,False
2,2,created_at,TIMESTAMP WITH TIME ZONE,False,None,False
3,3,title,VARCHAR,False,None,False
4,4,company,VARCHAR,False,None,False
5,5,location,VARCHAR,False,None,False
6,6,url,VARCHAR,False,None,False
7,7,description,VARCHAR,False,None,False
8,8,extra_json,VARCHAR,False,None,False


In [4]:
con.execute("PRAGMA table_info('adzuna_raw_index')").df()

,cid,name,type,notnull,dflt_value,pk
0,0,download_date,DATE,False,None,False
1,1,timestamp_local,TIMESTAMP,False,None,False
2,2,url,VARCHAR,False,None,False
3,3,what,VARCHAR,False,None,False
4,4,where,VARCHAR,False,None,False
5,5,results_per_page,BIGINT,False,None,False
6,6,sort_by,VARCHAR,False,None,False
7,7,what_exclude,VARCHAR,False,None,False
8,8,count,BIGINT,False,None,False
9,9,n_results,BIGINT,False,None,False


In [5]:
con.execute("SELECT COUNT(*) AS n FROM jobs_entry_level_v1").df()

,n
0,293


In [6]:
con.execute("SELECT COUNT(*) AS n FROM adzuna_raw_index").df()

,n
0,17


In [7]:
con.execute("""
WITH sample AS (
  SELECT extra_json
  FROM jobs_entry_level_v1
  WHERE extra_json IS NOT NULL
),
keys AS (
  SELECT unnest(json_keys(extra_json)) AS k
  FROM sample
)
SELECT k, COUNT(*) AS freq_in_sample
FROM keys
GROUP BY k
ORDER BY freq_in_sample DESC, k
""").df()

,k,freq_in_sample
0,category_label,293
1,category_tag,293
2,desc_sig,293
3,fetched_at,293
4,location_area,293
5,queries,293
6,raw_request_file,293
7,raw_response_file,293
8,source_adref,293
9,source_salary_is_predicted,293


In [8]:
con.execute("""
SELECT
  json_type(json_extract(extra_json, '$.queries')) AS queries_type,
  json_extract(extra_json, '$.queries') AS queries_raw,
  json_type(json_extract(extra_json, '$.location_area')) AS location_area_type,
  json_extract(extra_json, '$.location_area') AS location_area_raw
FROM jobs_entry_level_v1
WHERE extra_json IS NOT NULL
LIMIT 10
""").df()

,queries_type,queries_raw,location_area_type,location_area_raw
0,ARRAY,"[""werkstudent data""]",ARRAY,"[""Deutschland"",""Berlin""]"
1,ARRAY,"[""werkstudent data""]",ARRAY,"[""Deutschland"",""Berlin""]"
2,ARRAY,"[""werkstudent data""]",ARRAY,"[""Deutschland"",""Baden-Württemberg"",""Ludwigsbur..."
3,ARRAY,"[""werkstudent data""]",ARRAY,"[""Deutschland""]"
4,ARRAY,"[""werkstudent data""]",ARRAY,"[""Deutschland"",""Baden-Württemberg"",""Pforzheim""..."
5,ARRAY,"[""werkstudent data""]",ARRAY,"[""Deutschland"",""Bayern"",""Ingolstadt"",""Unterhau..."
6,ARRAY,"[""werkstudent data""]",ARRAY,"[""Deutschland"",""Bremen""]"
7,ARRAY,"[""werkstudent data""]",ARRAY,"[""Deutschland"",""Bayern"",""München (Kreis)"",""Mün..."
8,ARRAY,"[""werkstudent data""]",ARRAY,"[""Deutschland"",""Hessen"",""Frankfurt am Main"",""B..."
9,ARRAY,"[""werkstudent data""]",ARRAY,"[""Deutschland"",""Hamburg"",""Finkenwerder""]"


In [9]:
con.execute("""
SELECT
  job_id,
  json_extract_string(extra_json, '$.queries[0]') AS channel0,
  json_extract_string(extra_json, '$.location_area[1]') AS bundesland_guess,
  json_array_length(json_extract(extra_json, '$.location_area')) AS location_area_len
FROM jobs_entry_level_v1
WHERE extra_json IS NOT NULL
LIMIT 10
""").df()

,job_id,channel0,bundesland_guess,location_area_len
0,5555029867,werkstudent data,Berlin,2
1,5555029872,werkstudent data,Berlin,2
2,5477124023,werkstudent data,Baden-Württemberg,4
3,4931269716,werkstudent data,None,1
4,5453154769,werkstudent data,Baden-Württemberg,4
5,5518285192,werkstudent data,Bayern,4
6,5503646023,werkstudent data,Bremen,2
7,5453165641,werkstudent data,Bayern,5
8,4931261452,werkstudent data,Hessen,4
9,5555599124,werkstudent data,Hamburg,3


In [10]:
con.execute("""
SELECT
  json_array_length(json_extract(extra_json, '$.queries')) AS queries_len,
  COUNT(*) AS n_rows
FROM jobs_entry_level_v1
GROUP BY 1
ORDER BY 1
""").df()


,queries_len,n_rows
0,1,292
1,2,1


In [11]:
con.execute("""
SELECT
  j.job_id,
  json_extract_string(q.value, '$') AS channel
FROM jobs_entry_level_v1 j,
     json_each(j.extra_json, '$.queries') q
LIMIT 10
""").df()


,job_id,channel
0,5555029867,werkstudent data
1,5555029872,werkstudent data
2,5477124023,werkstudent data
3,4931269716,werkstudent data
4,5453154769,werkstudent data
5,5518285192,werkstudent data
6,5503646023,werkstudent data
7,5453165641,werkstudent data
8,4931261452,werkstudent data
9,5555599124,werkstudent data


# Views

In [27]:
con.execute(Path(ROOT/"sql/semantic/v_jobs.sql").read_text(encoding="utf-8"))
con.execute("""
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT job_id) AS unique_jobs,
  COUNT(DISTINCT channel) AS unique_channels,
  SUM(CASE WHEN bundesland IS NULL THEN 1 ELSE 0 END) AS bundesland_nulls
FROM v_jobs
""").df()

,rows,unique_jobs,unique_channels,bundesland_nulls
0,294,293,3,47.0


In [28]:
con.execute(Path(ROOT/"sql/semantic/v_replication_groups.sql").read_text(encoding="utf-8"))
con.execute("SELECT COUNT(*) AS n FROM v_replication_groups").df()

,n
0,222


In [14]:
con.execute("""
SELECT company, channel, distinct_locations, postings, sample_title, sample_location
FROM v_replication_groups
ORDER BY distinct_locations DESC, postings DESC
LIMIT 10
""").df()

,company,channel,distinct_locations,postings,sample_title,sample_location
0,Westnetz GmbH,werkstudent data,8,8,Werkstudent Data Science (m/w/d),"Altendorf, Essen"
1,Simon-Kucher & Partners,praktikum data,6,6,Praktikum Data Engineer im Bereich Advanced An...,"Berlin, Deutschland"
2,Reply Deutschland SE,junior data,4,4,Junior Data Engineer (m/w/d) - Datenbankentwic...,"Bremen, Deutschland"
3,IBM,junior data,3,6,Senior Data Scientist - Public Sektor (m/w/d),Deutschland
4,Reply Deutschland SE,junior data,3,3,Junior Data Engineer (m/w/d) - Datenbankentwic...,"Berlin, Deutschland"
5,Simon-Kucher & Partners,praktikum data,3,3,Praktikum Associate Consultant Data Science (m...,"Berlin, Deutschland"
6,E.ON Country Hub Germany GmbH,werkstudent data,2,14,Student BWL als Studentische Hilfskraft HR Das...,"Berlin, Deutschland"
7,"ServiceNow, Inc.",junior data,2,4,Staff Database Administrator,"Berlin, Deutschland"
8,PAUL HARTMANN AG,werkstudent data,2,4,Student Data Science als Werkstudent - Entwick...,"Gunzenhausen, Weißenburg-Gunzenhausen (Kreis)"
9,The Stepstone Group GmbH,werkstudent data,2,3,Werkstudent (m/w/d) Data Quality,"Düsseldorf, Nordrhein-Westfalen"
